In [2]:
# Defining project paths
BASE_DIR = "D:/Capstone/capstone_repo"
DATA_DIR = f"{BASE_DIR}/data"
NB_DIR = f"{BASE_DIR}/notebooks"

import os
os.makedirs(DATA_DIR, exist_ok=True)

In [55]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import DBSCAN


In [41]:
def wrangle(df):
    df_clean = df[["name", 'lat', 'lon']].copy()
    df_clean = df_clean.dropna(subset=['lat', 'lon'])
    # Convert to GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df_clean,
        geometry=gpd.points_from_xy(df_clean.lon, df_clean.lat),
        crs="EPSG:4326"
    )
    
    return df_clean, gdf

In [42]:
def save_clean_data(df, gdf, filename):
    filepath = os.path.join(f"{DATA_DIR}/processed", filename)
    df.to_csv(filepath + ".csv", index=False)
    gdf.to_file(filepath + ".gpkg", layer="stops", driver="GPKG")

# Hospitals data

In [43]:
hospitals = pd.read_csv(f"{DATA_DIR}/raw/casablanca_healthcare.csv")
hospitals.head()

,name,amenity,geometry,lat,lon
0,Clinique Badr مصحة بدر,clinic,POINT (-7.6410981 33.5948697),33.594870,-7.641098
1,clinique dentaire casablanca (cdc),clinic,POINT (-7.6482094 33.5874104),33.587410,-7.648209
2,Centre consultation et traitement dentaires,hospital,POINT (-7.6206013 33.5730773),33.573077,-7.620601
3,Clinique d'accouchement,clinic,POINT (-7.6240936 33.5930596),33.593060,-7.624094
4,Hôpital Sidi Othmane,hospital,POINT (-7.5733008 33.5585424),33.558542,-7.573301


In [44]:
# Manually filling in missing names based on the original dataset and Google Maps to ensure accuracy. 
hospitals.loc[47, "name"] = "Clinique Al Oumouma"
hospitals.loc[71, "name"] = "HOPITAL UNIVERSITAIRE DE PROXIMITE"
hospitals.loc[80, "name"] = "Hôpital Municipal sidi moumen Tacharouk"
hospitals.loc[88, "name"] = "Centre de Maladies du Rein et de Dialyse AL AMINE"
hospitals.loc[97, "name"] = "مستشفى الوفاء"
hospitals.loc[98, "name"] = "المركز الصحي المسيرة 2 (المستوى التاني)"

### Filtering Non-Hospital Healthcare Facilities (Dental, Ophtalmo, etc.)

Dropping any remaining rows with missing names, as they cannot be reliably identified.

In [45]:
# List of keywords to exclude
keywords = [
    "dentaire", "accouchement", "maternité", "pharmacie",
    "labo", "kine", "kiné", "ophtal",
    "hemo", "cardio", "radio",
    "Al Oumouma", "Dialyse"
]

# Create regex pattern
pattern = "|".join(keywords)

# Drop null names
hospitals = hospitals.dropna(subset=["name"])

# Filter rows
hospitals = hospitals[~hospitals["name"].str.contains(pattern, case=False, na=False)]

In [46]:
hospitals[hospitals.duplicated(subset=["name"], keep=False)]

,name,amenity,geometry,lat,lon
7,Avicenne ابن سينا,clinic,POINT (-7.6110661 33.5542874),33.554287,-7.611066
8,Avicenne ابن سينا,clinic,POINT (-7.6109113 33.5543506),33.554351,-7.610911
54,Hopital 20 Aout,hospital,"POLYGON ((-7.6207405 33.5752817, -7.6206681 33...",33.575137,-7.621018
55,Hopital 20 Aout,hospital,"POLYGON ((-7.6217004 33.57581, -7.6183329 33.5...",33.574653,-7.619653


Manually checked the accuracy of the location of each duplicated hospital. Kept the one with the most accurate location

In [47]:
hospitals = hospitals.drop_duplicates(subset=["name"], keep="last")

In [48]:
healthcare_cleaned, healthcare_gdf = wrangle(hospitals)

In [49]:
save_clean_data(healthcare_cleaned, healthcare_gdf, "Casablanca_Healthcare")

# Public Transport data: OSM Data

In [37]:
transport_stops = pd.read_csv(f"{DATA_DIR}/raw/casablanca_transport_stops.csv")
transport_stops.head()

,name,public_transport,geometry,lat,lon
0,Marjane Drissia M6,platform,POINT (-7.5946468 33.5618694),33.561869,-7.594647
1,Usine de Thé,platform,POINT (-7.5161146 33.6013266),33.601327,-7.516115
2,Sakani,platform,POINT (-7.4821131 33.6036527),33.603653,-7.482113
3,Résidence El Hamd,platform,POINT (-7.5424947 33.5573105),33.557310,-7.542495
4,Groupe Scolaire La Fourmilière,platform,POINT (-7.665272 33.5805828),33.580583,-7.665272


In [38]:
transport_stops["public_transport"].value_counts()

public_transport
platform         1616
stop_position     177
Name: count, dtype: int64

In [39]:
transport_stops_cleaned = wrangle(transport_stops)
transport_stops_cleaned.shape

(1793, 3)

In [40]:
save_clean_data(transport_stops_cleaned, "casablanca_transport_stops_cleaned.csv")

# Excel Healthcare data

In [89]:
import openpyxl 

wb = openpyxl.load_workbook(f'{DATA_DIR}/raw/liste-des-hopitaux-ms.xlsx')
ws = wb.active

In [90]:
header = [ws.cell(row=3,column=i).value for i in range(1,6)]
print(header)

['Région', 'Delegation', 'Commune', 'Etablissement hospitalier', 'Catégorie']


In [91]:
# reading data
my_list = list()

for value in ws.iter_rows(
    min_row=4, max_row=ws.max_row, min_col=1, max_col=5, 
    values_only=True):
    my_list.append(value)

In [92]:
df = pd.DataFrame(my_list, columns=header)
df.head()

,Région,Delegation,Commune,Etablissement hospitalier,Catégorie
0,Tanger-Tetouan-Al Hoceima,Al Hoceima,Al Hoceima (Mun.),Mohamed V,HP
1,Tanger-Tetouan-Al Hoceima,Al Hoceima,Al Hoceima (Mun.),C. d'oncologie d'Al Hoceima,CRO
2,Tanger-Tetouan-Al Hoceima,Al Hoceima,Imzouren (Mun.),Imzouren,HPr
3,Tanger-Tetouan-Al Hoceima,Al Hoceima,Targuist (Mun.),Targuist,HPr
4,Tanger-Tetouan-Al Hoceima,Chefchaouen,Chefchaouen (Mun.),Mohamed V,HP


In [93]:
Casa_Delegation = ["Casablanca Anfa", "Al Fida-Mers Sultan", "Ain Sebaâ-Hay Mohammadi", "Ben Msick", "Sidi Bernoussi", "Moulay Rachid", "Mediouna", "Nouaceur", "Aïn Chok", "Hay Hassani"]
df_casa = df[df["Delegation"].isin(Casa_Delegation)]
df_casa.head()

,Région,Delegation,Commune,Etablissement hospitalier,Catégorie
97,Casablanca-Settat,Mediouna,Mediouna (Mun.),CHP Mediouna,HP
98,Casablanca-Settat,Mediouna,Sidi Hajjaj Oued Hassar,Hôpital Psychiatrique Arrazi,HPsyP
100,Casablanca-Settat,Nouaceur,Bouskoura (Mun.),Bouskoura,HPr
101,Casablanca-Settat,Nouaceur,Dar Bouazza (Mun.),Prince My Hassan,HP
106,Casablanca-Settat,Casablanca Anfa,El Maarif (Arrond.),20 Aout 1953,HIR


In [94]:
import re

def clean_text(text):
    text = re.sub(r"\(.*?\)", "", text)  # remove anything in parentheses
    return text.strip()

df_casa["Commune_clean"] = df_casa["Commune"].apply(clean_text)


C:\Users\afafb\AppData\Local\Temp\ipykernel_7152\407991542.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_casa["Commune_clean"] = df_casa["Commune"].apply(clean_text)


In [95]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time


# Initialize geocoder
geolocator = Nominatim(user_agent="capstone_healthcare_project")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Create full address column
df_casa["address"] = (
    df_casa["Etablissement hospitalier"] + ", " +
    df_casa["Commune"] + ", Morocco"
)


C:\Users\afafb\AppData\Local\Temp\ipykernel_7152\3956227501.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_casa["address"] = (


In [96]:
# # Apply geocoding
# df_casa["location"] = df_casa["full_address"].apply(geocode)

# # Extract lat/lon
# df_casa["latitude"] = df_casa["location"].apply(lambda loc: loc.latitude if loc else None)
# df_casa["longitude"] = df_casa["location"].apply(lambda loc: loc.longitude if loc else None)

# df_casa.drop(columns=["location"], inplace=True)

# df_casa.head()

In [97]:
import sys
import os

# add the repo root to the path
sys.path.append(os.path.abspath(".."))

from config import GCP_API_KEY


In [99]:
import pandas as pd
import requests

API_KEY = GCP_API_KEY

def geocode_google(address):
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={API_KEY}"
    response = requests.get(url).json()
    if response['status'] == 'OK':
        location = response['results'][0]['geometry']['location']
        return location['lat'], location['lng']
    else:
        return None, None

df_casa['latitude'], df_casa['longitude'] = zip(*df_casa['address'].apply(geocode_google))


C:\Users\afafb\AppData\Local\Temp\ipykernel_7152\3317024188.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_casa['latitude'], df_casa['longitude'] = zip(*df_casa['address'].apply(geocode_google))
C:\Users\afafb\AppData\Local\Temp\ipykernel_7152\3317024188.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_casa['latitude'], df_casa['longitude'] = zip(*df_casa['address'].apply(geocode_google))


In [111]:
save_clean_data(df_casa, "casablanca_hospitals_geocoded.csv")

In [101]:
hospitals_osm = pd.read_csv(f"{DATA_DIR}/processed/casablanca_healthcare_cleaned.csv")
hospitals_osm.head()

,name,lat,lon
0,Centre consultation et traitement dentaires,33.573077,-7.620601
1,Hôpital Sidi Othmane,33.558542,-7.573301
2,Clinique les princes مصحة الأمراء,33.587862,-7.619570
3,Hospital CNSS,33.610752,-7.500804
4,Clinical Les Fleurs,33.581587,-7.622023


In [105]:
from rapidfuzz import process

def match_name(name, choices):
    match = process.extractOne(name, choices)
    return match[0] if match else None

In [108]:
ministry_names = df_casa["Etablissement hospitalier"].tolist()
osm_names = hospitals_osm["name"].tolist()

df_casa["matched_osm_name"] = df_casa["Etablissement hospitalier"].apply(
    lambda x: match_name(x, osm_names)
)

C:\Users\afafb\AppData\Local\Temp\ipykernel_7152\3671428568.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_casa["matched_osm_name"] = df_casa["Etablissement hospitalier"].apply(


In [110]:
df_casa.head()

,Région,Delegation,Commune,Etablissement hospitalier,Catégorie,Commune_clean,address,latitude,longitude,matched_osm_name
97,Casablanca-Settat,Mediouna,Mediouna (Mun.),CHP Mediouna,HP,Mediouna,"CHP Mediouna, Mediouna (Mun.), Morocco",33.459781,-7.517841,Clinique Mersultan
98,Casablanca-Settat,Mediouna,Sidi Hajjaj Oued Hassar,Hôpital Psychiatrique Arrazi,HPsyP,Sidi Hajjaj Oued Hassar,"Hôpital Psychiatrique Arrazi, Sidi Hajjaj Oued...",33.528972,-7.436498,Hôpital Privé International de Casablanca المس...
100,Casablanca-Settat,Nouaceur,Bouskoura (Mun.),Bouskoura,HPr,Bouskoura,"Bouskoura, Bouskoura (Mun.), Morocco",33.457363,-7.650380,Hôpital AL mansour
101,Casablanca-Settat,Nouaceur,Dar Bouazza (Mun.),Prince My Hassan,HP,Dar Bouazza,"Prince My Hassan, Dar Bouazza (Mun.), Morocco",33.524034,-7.825917,Polyclinique CNSS Hay Hassani مصحة الضمان الإج...
106,Casablanca-Settat,Casablanca Anfa,El Maarif (Arrond.),20 Aout 1953,HIR,El Maarif,"20 Aout 1953, El Maarif (Arrond.), Morocco",33.574695,-7.619993,Hopital 20 Aout


# Public Transport: Collected via JSON API

In [50]:
stops = pd.read_csv(f"{DATA_DIR}/raw/CasaBus/CasaBus_stops.csv")
stops.head()

,StopId,StopName,DirectionId,RouteId,LigneName,RouteColor,RouteTextColor,StopLat,StopLon
0,0:112 Boulevard D'Anfa,112 Boulevard D'Anfa,1,13,L013,1FE6E6,000000,33.592059,-7.633955
1,0:112 Boulevard D'Anfa,112 Boulevard D'Anfa,1,50,L050,990033,FFFFFF,33.592059,-7.633955
2,0:112 Boulevard D'Anfa,112 Boulevard D'Anfa,1,84,L084,0033FF,FFFFFF,33.592059,-7.633955
3,0:112 Boulevard D'Anfa,112 Boulevard D'Anfa,1,9E,L09E,0033FF,FFFFFF,33.592059,-7.633955
4,0:112 Boulevard D'Anfa_,112 Boulevard D'Anfa_,0,13,L013,1FE6E6,000000,33.592137,-7.633524


In [59]:
# remove trailing underscore
stops["StopName"] = stops["StopName"].str.rstrip("_")

# optional: standardize
stops["StopName"] = stops["StopName"].str.strip().str.upper()

In [60]:
unique_stops = stops.groupby(
    ["StopName", "StopLat", "StopLon"],
    as_index=False
).agg({
    "RouteId": "nunique"
})

unique_stops.rename(columns={"RouteId": "NumRoutes"}, inplace=True)

In [61]:
# Convert to GeoDataFrame
stops_gdf = gpd.GeoDataFrame(
    unique_stops,
    geometry=gpd.points_from_xy(unique_stops.StopLon, unique_stops.StopLat),
    crs="EPSG:4326"
)

In [62]:
stops_gdf = stops_gdf.to_crs(epsg=32629)

In [63]:
# extract coordinates in meters
coords = np.array(list(zip(stops_gdf.geometry.x, stops_gdf.geometry.y)))

# cluster within 20 meters
clustering = DBSCAN(eps=20, min_samples=1).fit(coords)

stops_gdf["cluster"] = clustering.labels_

In [65]:
merged_stops = stops_gdf.groupby("cluster").agg({
    "StopName": "first",
    "NumRoutes": "nunique",
    "geometry": "centroid"
}).reset_index()


merged_stops = gpd.GeoDataFrame(merged_stops, geometry="geometry", crs="EPSG:32629")

AttributeError: 'SeriesGroupBy' object has no attribute 'centroid'

In [ ]:
merged_stops = merged_stops.to_crs(epsg=4326)

In [54]:
save_clean_data(stops_cleaned, stops_gdf, "CasaBus_stops")